### Python Descriptors

***Descriptors*** are Python objects that implement a method of the ***descriptor protocol***, which gives you the ability to create objects that have special behavior when they’re accessed as attributes of other objects.

* `__get__(self, obj, type=None) -> object`
* `__set__(self, obj, value) -> None`
* `__delete__(self, obj) -> None`
* `__set_name__(self, owner, name)`

If your descriptor implements just `.__get__()`, then it’s said to be a ***non-data descriptor***. If it implements `.__set__()` or `.__delete__()`, then it’s said to be a ***data descriptor***. Note that this difference is not just about the name, but it’s also a difference in behavior. That’s because data descriptors have precedence during the lookup process

a descriptor that logs something on the console when it’s accessed:

In [2]:
class Verbose_attribute:
    def __get__(self, obj, type=None) -> object:
        print("accessing the attribute to get the value")
        return 42
    
    def __set__(self, obj, value) -> None:
        print("accessing the attribute to set the value")
        raise AttributeError("Cannot change the value")

class Foo:
    attribute1 = Verbose_attribute()

my_foo_object = Foo()
x = my_foo_object.attribute1
print(x)

accessing the attribute to get the value
42


`Verbose_attribute()` implements the ***descriptor protocol***. Once it’s instantiated as an attribute of `Foo`, it can be considered a ***descriptor***.

As a descriptor, it has ***binding behavior*** when it’s accessed using ***dot notation***. In this case, the descriptor logs a message on the console every time it’s accessed to ***get or set*** a value:

* When it’s accessed to `.__get__()` the value, it always returns the value `42`.
* When it’s accessed to `.__set__()` a specific value, it raises an `AttributeError` exception, which is the recommended way to implement ***read-only descriptors***.

Here, when you try to access `attribute1`, the descriptor logs this access to the console, as defined in `.__get__()`.

### How Descriptors Work in Python’s Internals

#### Python Descriptors in Properties

In [3]:
class Foo:
    @property
    def attribute1(self) -> object:
        print("accessing the attribute to get the value")
        return 42

    @attribute1.setter
    def attribute1(self, value) -> None:
        print("accessing the attribute to set the value")
        raise AttributeError("Cannot change the value")

my_foo_object = Foo()
x = my_foo_object.attribute1
print(x)

accessing the attribute to get the value
42


this can be written as follows:

In [4]:
class Foo:
    def getter(self) -> object:
        print("accessing the attribute to get the value")
        return 42

    def setter(self, value) -> None:
        print("accessing the attribute to set the value")
        raise AttributeError("Cannot change the value")

    attribute1 = property(fget=getter, fset=setter)

my_foo_object = Foo()
x = my_foo_object.attribute1
print(x)

accessing the attribute to get the value
42


`property(fget=None, fset=None, fdel=None, doc=None) -> object`

`property()` returns a property object that implements the ***descriptor protocol***. It uses the parameters `fget`, `fset` and `fdel` for the actual implementation of the three methods of the protocol.

#### Python Descriptors in Methods and Functions

The magic that transforms your `obj.method(*args)` call into `method(obj, *args)` is inside a `.__get__()` implementation of the function object that is, in fact, a ***non-data descriptor***. In particular, the function object implements `.__get__()` so that it returns a bound method when you access it with dot notation. The `(*args)` that follow invoke the functions by passing all the extra arguments needed.

In [1]:
import types

class Function(object):
    ...
    def __get__(self, obj, objtype=None):
        "Simulate func_descr_get() in Objects/funcobject.c"
        if obj is None:
            return self
        return types.MethodType(self, obj)

when the function is accessed with dot notation, `.__get__()` is called and a bound method is returned.

This works for regular instance methods just like it does for class methods or static methods. So, if you call a static method with `obj.method(*args)`, then it’s automatically transformed into `method(*args)`. Similarly, if you call a class method with `obj.method(type(obj), *args)`, then it’s automatically transformed into `method(type(obj), *args)`

a possible static method implementation could be this:

In [2]:
class StaticMethod(object):
    "Emulate PyStaticMethod_Type() in Objects/funcobject.c"
    def __init__(self, f):
        self.f = f

    def __get__(self, obj, objtype=None):
        return self.f

Likewise, this could be a possible class method implementation:

In [3]:
class ClassMethod(object):
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"
    def __init__(self, f):
        self.f = f

    def __get__(self, obj, klass=None):
        if klass is None:
            klass = type(obj)
        def newfunc(*args):
            return self.f(klass, *args)
        return newfunc

Note that, in Python, a class method is just a static method that takes the class reference as the first argument of the argument list.

### How Attributes Are Accessed With the Lookup Chain

In Python, every object has a built-in `__dict__` attribute. This is a dictionary that contains all the attributes defined in the object itself. To see this in action, consider the following example:

In [2]:
class Vehicle:
    can_fly = False
    number_of_weels = 0

class Car(Vehicle):
    number_of_weels = 4

    def __init__(self, color):
        self.color = color

In [3]:
my_car = Car("red")

In [5]:
print(my_car.__dict__)  # instance namespace
print(type(my_car).__dict__)  # class namespace

{'color': 'red'}
{'__module__': '__main__', 'number_of_weels': 4, '__init__': <function Car.__init__ at 0x000001FAE2FCA200>, '__doc__': None}


***when you access an attribute in Python***

In [6]:
class Vehicle(object):
    can_fly = False
    number_of_weels = 0

class Car(Vehicle):
    number_of_weels = 4

    def __init__(self, color):
        self.color = color

In [7]:
my_car = Car("red")

In [9]:
print(my_car.color)  # accessing a single value of the __dict__ attribute of the object 'my_car'
print(my_car.number_of_weels)  # accessing a single value of the __dict__ attribute of the class 'Car'
print(my_car.can_fly)  # accessing using the __dict__ attribute of the 'Vehicle' class

red
4
False


This means that it’s possible to rewrite the above example like this:

In [10]:
class Vehicle:
    can_fly = False
    number_of_weels = 0

class Car(Vehicle):
    number_of_weels = 4

    def __init__(self, color):
        self.color = color

In [11]:
my_car = Car("red")

In [12]:
print(my_car.__dict__['color'])
print(type(my_car).__dict__['number_of_weels'])
print(type(my_car).__base__.__dict__['can_fly'])

red
4
False


How does the interpreter know what you really need?

***the lookup chain:***
* First, you’ll get the result returned from the `__get__` method of the ***data descriptor*** named after the attribute you’re looking for;
* If that fails, then you’ll get the value of your object’s `__dict__` for the key named after the attribute you’re looking for;
* If that fails, then you’ll get the result returned from the `__get__` method of the ***non-data descriptor*** named after the attribute you’re looking for;
* If that fails, then you’ll get the value of your object type’s `__dict__` for the key named after the attribute you’re looking for;
* If that fails, then you’ll get the value of your object parent type’s `__dict__` for the key named after the attribute you’re looking for;
* If that fails, then the previous step is repeated for ***all the parent’s types*** in the method resolution order of your object
* If everything else has failed, then you’ll get an `AttributeError` exception.

Now you can see why it’s important to know if a descriptor is a ***data descriptor*** or a ***non-data descriptor***? They’re on different levels of the lookup chain, and you’ll see later on that this difference in behavior can be very convenient.

### How to Use Python Descriptors Properly

If you want to use Python descriptors in your code, then you just need to implement the ***descriptor protocol***. The most important methods of this protocol are `.__get__()` and `.__set__()`, which have the following signature:

* `__get__(self, obj, type=None) -> object`
* `__set__(self, obj, value) -> None`

* `self` is the instance of the descriptor you’re writing;
* `obj` is the instance of the object your descriptor is attached to;
* `type` is the type of the object the descriptor is attached to.

In `.__set__()`, you don’t have the `type` variable, because you can only call `.__set__()` on the object. In contrast, you can call `.__get__()` on both the object and the class.

Another important thing to know is that Python descriptors are instantiated ***just once*** per class.

*That means that every single instance of a class containing a descriptor ***shares*** that descriptor instance*. This is something that you might not expect and can lead to a classic pitfall, like this:

In [2]:
class OneDigitNumericValue:
    def __init__(self):
        self.value = 0
        
    def __get__(self, obj, type=None) -> object:
        return self.value
    
    def __set__(self, obj, value) -> None:
        if value > 9 or value < 0 or int(value) != value:
            raise AttributeError("The value is invalid")
        self.value = value

class Foo:
    number = OneDigitNumericValue()  # class attribte is commom for all instances

In [3]:
my_foo_object = Foo()
my_second_foo_object = Foo()

In [5]:
my_foo_object.number = 3  # my_foo_object doesn't have attibute 'number', so, Python goes to class scope to get it and modify

In [6]:
print(my_foo_object.number)
print(my_second_foo_object.number)

3
3


class attibute `number` has been modified now

In [7]:
my_third_foo_object = Foo()

In [8]:
print(my_third_foo_object.number)

3


Here, you have a class `Foo` that defines an attribute `number`, which is a descriptor. This descriptor accepts a single-digit numeric value and stores it in a property of the descriptor itself. However, this approach won’t work, because each instance of `Foo` shares the same descriptor instance. What you’ve essentially created is just a new ***class-level attribute***.

all the instances of `Foo` have the same value for the attribute `number`, even though the last one was created after the `my_foo_object.number` attribute was set

So, how can you solve this problem?

1. use a dictionary to save all the ***values*** of the descriptor for all the ***objects*** it’s attached to. `.__get__()` and `.__set__()` have the `obj` attribute, which is the instance of the object you’re attached to. You could use this value as a key for the dictionary:

In [19]:
class OneDigitNumericValue:
    def __init__(self):
        self.value = {}

    def __get__(self, obj, type=None) -> object:
        try:
            return self.value[obj]
        except:
            return 0

    def __set__(self, obj, value) -> None:
        if value > 9 or value < 0 or int(value) != value:
            raise AttributeError("The value is invalid")
        self.value[obj] = value

class Foo():
    number = OneDigitNumericValue()

In [20]:
my_foo_object = Foo()
my_second_foo_object = Foo()

In [21]:
my_foo_object.number = 3

In [22]:
print(my_foo_object.number)
print(my_second_foo_object.number)

3
0


In [29]:
my_third_foo_object = Foo()

In [30]:
print(my_third_foo_object.number)

0


we use a dictionary for storing the value of the `number` attribute for all objects inside the descriptor

the downside here is that the descriptor is keeping a ***strong reference*** to the owner object. This means that if you destroy the object, then the memory is not released because the garbage collector keeps finding a reference to that object inside the descriptor!

You may think that the solution here could be the use of ***weak references***. While that may, you’d have to deal with the fact that not everything can be referenced as weak and that, when your objects get collected, they disappear from your dictionary.

The best solution here is to simply not store values in the descriptor itself, but to store them in the object that the descriptor is attached to. Try this approach next:

In [9]:
class OneDigitNumericValue:
    def __init__(self, name):
        self.name = name

    def __get__(self, obj, type=None) -> object:
        return obj.__dict__.get(self.name) or 0

    def __set__(self, obj, value) -> None:
        obj.__dict__[self.name] = value  # store the value in the object dict

class Foo():
    number = OneDigitNumericValue("number")  # we have to instantiate a name for descriptor

In [10]:
my_foo_object = Foo()
my_second_foo_object = Foo()

In [11]:
my_foo_object.number = 3

In [12]:
print(my_foo_object.number)
print(my_second_foo_object.number)

3
0


In [13]:
vars(my_foo_object)

{'number': 3}

In [14]:
vars(my_second_foo_object)

{}

In [16]:
vars(type(my_foo_object))

mappingproxy({'__module__': '__main__',
              'number': <__main__.OneDigitNumericValue at 0x12998d239d0>,
              '__dict__': <attribute '__dict__' of 'Foo' objects>,
              '__weakref__': <attribute '__weakref__' of 'Foo' objects>,
              '__doc__': None})

In [17]:
my_third_foo_object = Foo()

In [18]:
print(my_third_foo_object.number)

0


In this example, when you set a value to the number attribute of your object, the descriptor stores it in the `__dict__` attribute of the object it’s attached to using the same name of the descriptor itself.

The only problem here is that when you instantiate the descriptor you have to specify the name as a parameter:
* `number = OneDigitNumericValue("number")`

Wouldn’t it be better to just write `number = OneDigitNumericValue()`? It might, but if you’re running a version of Python less than 3.6, then you’ll need a little bit of magic here with metaclasses and decorators. If you use Python 3.6 or higher, however, then the ***descriptor protocol*** has a new method `.__set_name__()` that does all this magic for you, as proposed in PEP 487:

`__set_name__(self, owner, name)`

***With this new method, whenever you instantiate a descriptor this method is called and the `name` parameter automatically set.***

In [19]:
class OneDigitNumericValue:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, type=None) -> object:
        return obj.__dict__.get(self.name) or 0

    def __set__(self, obj, value) -> None:
        obj.__dict__[self.name] = value

class Foo():
    number = OneDigitNumericValue()

In [20]:
my_foo_object = Foo()
my_second_foo_object = Foo()

In [21]:
my_foo_object.number = 3

In [22]:
print(my_foo_object.number)
print(my_second_foo_object.number)

3
0


In [23]:
my_third_foo_object = Foo()

In [24]:
print(my_third_foo_object.number)

0


`.__init__()` has been removed and `.__set_name__()` has been implemented. This makes it possible to create your descriptor without specifying the name of the internal attribute that you need to use for storing the value. Your code also looks nicer and cleaner now!

### Why Use Python Descriptors?

#### Lazy Properties

The first and most straightforward example is ***lazy properties***. These are properties whose initial values are not loaded until they’re accessed for the first time. Then, they load their initial value and keep that value cached for later reuse.

In [1]:
import time

class DeepThought:
    
    def meaning_of_life(self):
        time.sleep(3)
        return 42

In [2]:
my_deep_thought_instance = DeepThought()

In [3]:
print(my_deep_thought_instance.meaning_of_life())
print(my_deep_thought_instance.meaning_of_life())
print(my_deep_thought_instance.meaning_of_life())

42
42
42


a lazy property can instead evaluate this method just once when it’s first executed. Then, it will cache the resulting value so that, if you need it again, you can get it in no time. You can achieve this with the use of Python descriptors:

In [28]:
import time

class LazyProperty:
    def __init__(self, function):
        self.function = function  # 1. store 'meaning_of_life()'
        self.name = function.__name__  # 1. store 'meaning_of_life' method name

    def __get__(self, obj, type=None) -> object:
        obj.__dict__[self.name] = self.function(obj)  # 2. store resulting value
        return obj.__dict__[self.name]

class DeepThought:
    @LazyProperty
    def meaning_of_life(self):
        time.sleep(3)
        return 42

In [29]:
my_deep_thought_instance = DeepThought()

In [30]:
print(my_deep_thought_instance.meaning_of_life)  # 2. store resulting value
print(my_deep_thought_instance.meaning_of_life)  # 3. get the value from cached values in '__dict__'
print(my_deep_thought_instance.meaning_of_life)

42
42
42


* 1. when you use the `@LazyProperty` descriptor, you’re instantiating a descriptor and passing to it `.meaning_of_life()`. This descriptor stores both the method and its name as instance variables;
* 2. Since it is a ***non-data descriptor***, when you first access the value of the `meaning_of_life attribute`, `.__get__()` is automatically called and executes `.meaning_of_life()` on the `my_deep_thought_instance` object. The resulting value is stored in the `__dict__` attribute of the object itself;
* 3. When you access the `meaning_of_life` attribute again, Python will use the ***lookup chain*** to find a value for that attribute inside the `__dict__` attribute, and that value will be returned immediately.

#### Note that this works because, in this example, you’ve only used one method `.__get__()` of the descriptor protocol. You’ve also implemented a ***non-data descriptor***. If you had implemented a ***data descriptor***, then the trick would not have worked.

Following the lookup chain, it would have had precedence over the value stored in `__dict__`. To test this out, run the following code::

In [31]:
import time

class LazyProperty:
    def __init__(self, function):
        self.function = function
        self.name = function.__name__

    def __get__(self, obj, type=None) -> object:
        obj.__dict__[self.name] = self.function(obj)
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        pass

class DeepThought:
    @LazyProperty
    def meaning_of_life(self):
        time.sleep(3)
        return 42

In [32]:
my_deep_thought_instance = DeepThought()

In [33]:
print(my_deep_thought_instance.meaning_of_life)
print(my_deep_thought_instance.meaning_of_life)
print(my_deep_thought_instance.meaning_of_life)

42
42
42


In this example, you can see that just implementing `.__set__()`, even if it doesn’t do anything at all, creates a ***data descriptor***. Now, the trick of the lazy property stops working.

#### D.R.Y. Code

Python descriptors give developers a great tool to write reusable code that can be shared among different properties or even different classes.

Consider an example where you have five different properties with the same behavior. Each property can be set to a specific value only if it’s an even number. Otherwise, it’s value is set to 0:

In [34]:
class Values:
    def __init__(self):
        self._value1 = 0
        self._value2 = 0
        self._value3 = 0
        self._value4 = 0
        self._value5 = 0

    @property
    def value1(self):
        return self._value1

    @value1.setter
    def value1(self, value):
        self._value1 = value if value % 2 == 0 else 0

    @property
    def value2(self):
        return self._value2

    @value2.setter
    def value2(self, value):
        self._value2 = value if value % 2 == 0 else 0

    @property
    def value3(self):
        return self._value3

    @value3.setter
    def value3(self, value):
        self._value3 = value if value % 2 == 0 else 0

    @property
    def value4(self):
        return self._value4

    @value4.setter
    def value4(self, value):
        self._value4 = value if value % 2 == 0 else 0

    @property
    def value5(self):
        return self._value5

    @value5.setter
    def value5(self, value):
        self._value5 = value if value % 2 == 0 else 0

In [35]:
my_values = Values()

In [36]:
my_values.value1 = 1
my_values.value2 = 4

In [37]:
print(my_values.value1)
print(my_values.value2)

0
4


As you can see, you have a lot of duplicated code here. It’s possible to use Python descriptors to share behavior among all the properties. You can create an `EvenNumber` descriptor and use it for all the properties like this:

In [39]:
class EvenNumber:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, type=None) -> object:
        return obj.__dict__.get(self.name) or 0

    def __set__(self, obj, value) -> None:
        obj.__dict__[self.name] = (value if value % 2 == 0 else 0)
        
class Values:
    value1 = EvenNumber()
    value2 = EvenNumber()
    value3 = EvenNumber()
    value4 = EvenNumber()
    value5 = EvenNumber()

In [40]:
my_values = Values()

In [41]:
my_values.value1 = 1
my_values.value2 = 4

In [42]:
print(my_values.value1)
print(my_values.value2)

0
4


This code looks a lot better now! The duplicates are gone and the logic is now implemented in a single place so that if you need to change it, you can do so easily.